# Real Estate Market Analysis in Germany

This notebook demonstrates the end-to-end workflow of the project:
- data processing
- data quality validation
- price prediction modeling
- evaluation
- export of final dataset

The data is based on listings scraped from Immowelt and enriched with external regional datasets.

In [5]:
%load_ext autoreload
%autoreload 2

In [1]:
import sys
import os
from pathlib import Path

current = Path().resolve()

# project root
while current != current.parent:
    if (current / 'src').exists():
        PROJECT_ROOT = current
        break
    current = current.parent

# for imports
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data/processed'

In [3]:
import pandas as pd

from src.pipeline import process_data
from src.model import run_modeling, classify_prices, add_metrics
from src.validation import create_data_quality_report

## Data Processing Pipeline

The pipeline includes:
- cleaning raw scraped data
- feature engineering
- enrichment with external datasets
- preparation of the final analytical dataset

In [4]:
df = process_data(RAW_DIR)
print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 9579 entries, 0 to 9578
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   id                          9579 non-null   int64   
 1   object_type                 9579 non-null   str     
 2   marketing_type              9579 non-null   str     
 3   is_valid                    9579 non-null   bool    
 4   price_euro                  9539 non-null   float64 
 5   size_sqm                    9538 non-null   float64 
 6   size_category               9538 non-null   category
 7   price_per_sqm               9501 non-null   float64 
 8   listings_per_1k_bundesland  8773 non-null   float64 
 9   listings_per_1k_city        5437 non-null   float64 
 10  density_city                8773 non-null   float64 
 11  density_land                8773 non-null   float64 
 12  population_land             8773 non-null   float64 
 13  location_full               9

## Data Quality Report

To validate the reliability of the dataset, data quality checks are applied.
The report includes completeness of key columns and logical validation flags.

In [5]:
report = create_data_quality_report(df)
report

,column,unique_values,filled_%,missing_%
0,id,9579,100.00,0.00
1,object_type,17,100.00,0.00
2,marketing_type,2,100.00,0.00
3,is_valid,2,100.00,0.00
4,price_euro,2572,99.58,0.42
5,size_sqm,1281,99.57,0.43
6,size_category,4,99.57,0.43
7,price_per_sqm,7067,99.19,0.81
8,listings_per_1k_bundesland,16,91.59,8.41
9,listings_per_1k_city,1047,56.76,43.24


## Price Prediction Model

A regression model is used to estimate price per square meter.
Separate logic is applied for rental and purchase listings.

In [6]:
df, _ = run_modeling(df)

In [7]:
df = add_metrics(df)
df = classify_prices(df)

## Model Evaluation

The model is evaluated using Mean Absolute Error (MAE).

In [8]:
mae_k = df.loc[df['marketing_type'] == 'zum Kauf']['MAE'].iloc[0]
mae_m = df.loc[df['marketing_type'] == 'zur Miete']['MAE'].iloc[0]
print(f"For renting MAE is: {mae_m:.2f} €/sqm")
print(f"For buying MAE is: {mae_k:.2f} €/sqm")

For renting MAE is: 3.11 €/sqm
For buying MAE is: 890.52 €/sqm


## Prediction Preview

The table below compares actual and predicted values.

In [9]:
df[[
    'marketing_type',
    'object_type',
    'price_per_sqm',
    'predicted_price',
    'predicted_price_per_sqm',
    'price_category'
]].head(10)

,marketing_type,object_type,price_per_sqm,predicted_price,predicted_price_per_sqm,price_category
0,zur Miete,Wohnung,18.571429,1750.278223,25.003975,Underpriced
1,zur Miete,Wohnung,12.375000,840.575634,10.507195,Fair Price
2,zur Miete,Wohnung,6.067251,506.306227,7.402138,Underpriced
3,zur Miete,Wohnung,6.500000,408.229370,7.850565,Underpriced
4,zur Miete,Wohnung,11.000000,1181.884080,13.132045,Fair Price
5,zur Miete,Wohnung,10.949464,1439.696019,11.023706,Fair Price
6,zum Kauf,Wohnung,6247.947455,350822.385026,5760.630296,Fair Price
7,zum Kauf,Wohnung,5952.033368,588915.342587,6140.931622,Fair Price
8,zur Miete,Wohnung,9.565217,461.504566,10.032708,Fair Price
9,zur Miete,Wohnung,12.195122,1084.007264,13.219601,Fair Price


## Export Final Dataset

The final processed dataset is exported for further analysis and Tableau visualization.

In [10]:
df.to_csv(
    PROCESSED_DIR / 'data_cleaned_sample1.csv',
    index=False,
    sep=';',
    encoding='utf-8-sig',
    decimal=','
)
print('Final dataset saved')

Final dataset saved


## Model Limitations

The model is intentionally simplified and does not include detailed location, building condition, or property quality features.
Therefore, predictions should be interpreted as approximate estimates rather than precise valuations.


## Practical Value

The final dataset and dashboards can be used to:
- compare regional market conditions
- identify overpriced listings
- analyze supply intensity across regions and cities